In [ ]:
import torch
# from dataset import DataLoader
# from model import GPT
# import from the classname you have created 

vocab_size = 50257
seq_len = 128         
num_dims = 256         
num_heads = 4          
num_layers = 6        

block_size = 128       
batch_size = 32        
device = "cuda" if torch.cuda.is_available() else "cpu"

learning_rate = 5e-4  
max_iters = 5000      
eval_interval = 250

dataset = DataLoader(block_size, batch_size, device)
model = GPT(vocab_size, seq_len, num_dims, num_heads, num_layers, p=0.1).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)


@torch.no_grad()
def estimate_loss(model, dataset, eval_iters=20):
    out = {}
    model.eval()

    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = dataset.get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()

    model.train()
    return out


best_val_loss = float("inf")

for i in range(max_iters):

    if i % eval_interval == 0 or i == max_iters - 1:
        losses = estimate_loss(model, dataset)
        print(f"Step {i:4d} | Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f}")
        
        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]
            checkpoint = {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "iter": i,
                "best_val_loss": best_val_loss,
            }
            torch.save(checkpoint, "best_checkpoint.pt")
            print(f"(Val Loss: {best_val_loss:.4f})")

    xb, yb = dataset.get_batch("train")

    optimizer.zero_grad(set_to_none=True)

    logits, loss = model(xb, yb)

    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()

final_checkpoint = {
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "iter": max_iters,
    "best_val_loss": best_val_loss,
}

torch.save(final_checkpoint, "latest_checkpoint.pt")
print("Training complete. Final state saved to latest_checkpoint.pt")

# Training Loop

So for the training of the model what do we need — the entire forward pass, and at the end of the model we return the `logits` and `loss`.

We need three objects:
- **Dataset object** — to get batches of `x` and `y`
- **Model object** — to do the forward pass
- **Optimizer object** — to update the weights

```python
dataset = DataLoader(block_size, batch_size, device)
model = GPT(vocab_size, seq_len, num_dims, num_heads, num_layers, p=0.1).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
```

---

## Estimate Loss

Next we need to set at how many step intervals we evaluate the model. That is `eval_interval`.

`eval_iters` is a bit different — it is how many steps we run during the evaluation to get a stable mean loss, not just a single noisy reading.

```python
@torch.no_grad()
def estimate_loss(model, dataset, eval_iters=20):
```

Yes, we do return `x` and `y` from `val_data` too. We are finding the loss from **both** train and val.

We loop through `["train", "val"]` — when the split is `"train"` the dataloader targets `train.bin`, when it is `"val"` it falls back to `val_data`. The naming is just for our convenience and consistency.

```python
for split in ["train", "val"]:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
        X, Y = dataset.get_batch(split)
        _, loss = model(X, Y)
        losses[k] = loss.item()
    out[split] = losses.mean()
```

When we call `model(X, Y)` we are sending `targets=Y`, so the loss gets computed inside the model. We ignore the logits here and only focus on the loss.

`losses` is a torch tensor of shape `(eval_iters,)` — it holds the loss for each of the 20 steps. We then sum it up and take the mean and store it in the `out` dict like:

```
out = {"train": mean_loss, "val": mean_loss}
```

`model.eval()` pauses the training behaviour (disables dropout etc.) and `model.train()` starts it again after evaluation is done.

---

## Training Loop

```python
best_val_loss = float("inf")
```

Before the loop starts, we initialize `best_val_loss` to infinity. This is how we track whether the model is actually getting better — every time the val loss drops below this we update it and save a new best checkpoint.

```python
for i in range(max_iters):

    if i % eval_interval == 0 or i == max_iters - 1:
        losses = estimate_loss(model, dataset)
        print(f"Step {i:4d} | Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f}")
```

At every `eval_interval` step we pause training, test the model on 20 steps for both train and val, and compute the means.

---

## Best Checkpoint — Only Save When It Actually Improves

```python
        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]
            checkpoint = {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "iter": i,
                "best_val_loss": best_val_loss,
            }
            torch.save(checkpoint, "best_checkpoint.pt")
```

We only save when the val loss is actually lower than the best we have seen so far. No point saving a worse model right?

The checkpoint dict saves not just the model weights but also the optimizer state, the current iteration, and the best val loss. This is important because if training crashes and we need to resume, we can restore exactly where we left off — the optimizer has its own internal state (momentum buffers, adaptive learning rates) that needs to be restored too, not just the model weights.

This saves to `best_checkpoint.pt` — only the best model ever seen during training.

---

## Training Step

```python
    xb, yb = dataset.get_batch("train")
    optimizer.zero_grad(set_to_none=True)
    logits, loss = model(xb, yb)
    loss.backward()
```

Get a batch, zero the gradients, forward pass, compute loss, backprop.

---

## Gradient Clipping

```python
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
```

Before the optimizer updates the weights, we clip the gradients. During backprop, gradients can sometimes explode — especially in deep transformers — and that would make the weight update massive and destabilize the whole training. `clip_grad_norm_` caps the global gradient norm at `1.0` so no single update can blow up the model.

---

## Final Save

```python
final_checkpoint = {
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "iter": max_iters,
    "best_val_loss": best_val_loss,
}
torch.save(final_checkpoint, "latest_checkpoint.pt")
```

After all `max_iters` are done, we save the final state to `latest_checkpoint.pt`. This is the most recent model — not necessarily the best one. So now we have two files:

- `best_checkpoint.pt` — the model at the point it had the lowest val loss
- `latest_checkpoint.pt` — the model at the very end of training

And this is where the difference actually matters.

Say we train for 5000 steps. At step 2500 the val loss hits `2.1` — the best it has ever been. But training keeps going and by step 5000 the val loss is sitting at `2.4`. The model got worse in the second half — maybe it started overfitting, maybe the learning rate was too high, whatever the reason.

If we only had `latest_checkpoint.pt`, we would be loading that worse `2.4` model. But because we tracked `best_val_loss` and saved separately, we can just load `best_checkpoint.pt` and get the `2.1` model from step 2500 — the one that actually performed the best.

```python
checkpoint = torch.load("best_checkpoint.pt")
model.load_state_dict(checkpoint["model"])
```

The model that finishes training is not always the best model. That is the whole point of tracking it separately.

> **Loop:** get batch → zero grad → forward → backward → clip → step → save best at interval → save final at end
